# Поисковая система по товарам (WANDS)
Bi-encoder для эмбеддингов → поиск по косинусной близости → реранкинг cross-encoder'ом.

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if not os.path.exists('product.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/product.csv', 'product.csv')

product = pd.read_csv('product.csv', sep='	')
query = pd.read_csv('query.csv', sep='	')
label = pd.read_csv('label.csv', sep='	')
product.shape, query.shape, label.shape

((42994, 9), (480, 3), (233448, 4))

In [2]:
print(device)

cuda


In [3]:
product.head(3)

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0


In [4]:
product = product.fillna('')
product['text'] = (product['product_name'] + '. ' + product['product_class'] + '. '
                   + product['category hierarchy'].str.replace('/', ' ') + '. '
                   + product['product_description'].str[:300])
product['text'].iloc[0]

'solid wood platform bed. Beds. Furniture   Bedroom Furniture   Beds & Headboards   Beds   Twin Beds. good , deep sleep can be quite difficult to have in this busy age . fortunately , there ’ s an antidote to such a problem : a nice , quality bed frame like the acacia kaylin . solidly constructed from acacia wood , this bed frame will stand the test of time and is fit to rest your shoulders on for y'

## Эмбеддинги

In [5]:
bi_encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

if os.path.exists('product_emb.npy'):
    product_emb = np.load('product_emb.npy')
else:
    product_emb = bi_encoder.encode(product['text'].tolist(), batch_size=128, normalize_embeddings=True,
                                    show_progress_bar=True, convert_to_numpy=True)
    np.save('product_emb.npy', product_emb)
product_emb.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(42994, 384)

## Поиск

In [6]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

def retrieve(q, k=100):
    q_emb = bi_encoder.encode(q, normalize_embeddings=True)
    scores = product_emb @ q_emb
    idx = np.argpartition(-scores, k)[:k]
    return idx[np.argsort(-scores[idx])]

def search(q, top_n=10, k=100, rerank=True):
    idx = retrieve(q, k)
    if rerank:
        ce_scores = cross_encoder.predict([(q, t) for t in product['text'].iloc[idx]])
        idx = idx[np.argsort(-ce_scores)]
    return product.iloc[idx[:top_n]][['product_id', 'product_name', 'product_class', 'average_rating']]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
search('salon chair')

,product_id,product_name,product_class,average_rating
25431,25431,barberpub salon massage chair,Massage Chairs,5.0
7465,7465,hair salon chair,Massage Chairs|Recliners,4.5
7468,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,5.0
25433,25433,barberpub hydraulic reclining massage chair,Massage Chairs|Recliners,4.5
24009,24009,gerster bar salon task chair,,
24010,24010,bar salon task chair,,
25432,25432,barberpub hydraulic salon spa reclining massag...,Massage Chairs|Recliners,4.0
7467,7467,reclining faux leather massage chair,Massage Chairs|Recliners,3.5
7466,7466,reclining massage chair,Massage Chairs|Recliners,2.0
24008,24008,germanton bar salon task chair,,


In [8]:
search('modern grey sofa with storage')

,product_id,product_name,product_class,average_rating
10836,10836,julianne 84 '' wide velvet reversible sleeper ...,,4.5
17649,17649,northstate mid-century modern light grey fabri...,Sofas,
1095,1095,oakman 75 '' pillow top arm reclining loveseat,Sofas,
17648,17648,ergenekon mid-century modern dark grey fabric ...,Sofas,
31141,31141,"convertible sectional sofa couch , l-shaped co...",Sectionals,4.0
6957,6957,samba charcoal grey upholstered recessed arm l...,Sofas,5.0
24005,24005,cilla 58 '' square arm loveseat,Sofas,4.5
42792,42792,gebhard storage convertible sofa,Futons,4.0
4018,4018,lelana 78.75 '' round arm sleeper,Futons|Sofas,4.0
31039,31039,minster tufted storage ottoman,,4.5


## Оценка качества: NDCG@10 по разметке WANDS

In [9]:
gain = label.assign(g=label['label'].map({'Exact': 2, 'Partial': 1, 'Irrelevant': 0}))
gain = gain.groupby('query_id').apply(lambda d: dict(zip(d['product_id'], d['g']))).to_dict()

def ndcg(pred_ids, rel, k=10):
    disc = 1 / np.log2(np.arange(2, k + 2))
    dcg = sum(rel.get(p, 0) * d for p, d in zip(pred_ids[:k], disc))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum(g * d for g, d in zip(ideal, disc))
    return dcg / idcg if idcg else 0

def evaluate(rerank):
    return np.mean([ndcg(search(q, rerank=rerank)['product_id'].tolist(), gain.get(qid, {}))
                    for qid, q in zip(query['query_id'], query['query'])])

print(f'bi-encoder:            {evaluate(False):.4f}')
print(f'bi-encoder + reranker: {evaluate(True):.4f}')

bi-encoder:            0.7031
bi-encoder + reranker: 0.7597


## E5-small-v2


In [10]:
e5 = SentenceTransformer('intfloat/e5-small-v2', device=device)

if os.path.exists('product_emb_e5.npy'):
    product_emb_e5 = np.load('product_emb_e5.npy')
else:
    product_emb_e5 = e5.encode(('passage: ' + product['text']).tolist(), batch_size=128, normalize_embeddings=True,
                               show_progress_bar=True, convert_to_numpy=True)
    np.save('product_emb_e5.npy', product_emb_e5)
product_emb_e5.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(42994, 384)

In [11]:
def search_e5(q, top_n=10, k=100, rerank=False):
    scores = product_emb_e5 @ e5.encode('query: ' + q, normalize_embeddings=True)
    idx = np.argpartition(-scores, k)[:k]
    idx = idx[np.argsort(-scores[idx])]
    if rerank:
        ce_scores = cross_encoder.predict([(q, t) for t in product['text'].iloc[idx]])
        idx = idx[np.argsort(-ce_scores)]
    return product.iloc[idx[:top_n]][['product_id', 'product_name', 'product_class', 'average_rating']]

In [12]:
search_e5('salon chair')

,product_id,product_name,product_class,average_rating
25431,25431,barberpub salon massage chair,Massage Chairs,5.0
7465,7465,hair salon chair,Massage Chairs|Recliners,4.5
25433,25433,barberpub hydraulic reclining massage chair,Massage Chairs|Recliners,4.5
25432,25432,barberpub hydraulic salon spa reclining massag...,Massage Chairs|Recliners,4.0
36910,36910,beauty spa salon barber chair,Massage Chairs,5.0
42928,42928,all purpose salon hydraulic barber massage chair,Massage Chairs|Recliners,4.0
39429,39429,barber shampoo salon reclining massage chair,Massage Chairs|Recliners,2.0
7468,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,5.0
22391,22391,hair beauty equipment hydraulic barber massage...,Massage Chairs,4.0
25430,25430,barberpub shampoo reclining massage chair,Massage Chairs|Recliners,3.5


In [13]:
search_e5('modern grey sofa with storage')

,product_id,product_name,product_class,average_rating
17649,17649,northstate mid-century modern light grey fabri...,Sofas,
17648,17648,ergenekon mid-century modern dark grey fabric ...,Sofas,
31141,31141,"convertible sectional sofa couch , l-shaped co...",Sectionals,4.0
23612,23612,allisa right facing sectional sofa . grey .,Sectionals,4.0
537,537,sectional sofa in gray linen,Sectionals,
24005,24005,cilla 58 '' square arm loveseat,Sofas,4.5
42792,42792,gebhard storage convertible sofa,Futons,4.0
12660,12660,chloe 74 '' square arm sofa,Sofas,4.0
26758,26758,garren 75.6 '' square arm sofa,Sofas,4.5
5841,5841,acker 86 '' square arm sofa,Sofas,5.0


In [14]:
def evaluate_fn(fn):
    return np.mean([ndcg(fn(q)['product_id'].tolist(), gain.get(qid, {}))
                    for qid, q in zip(query['query_id'], query['query'])])

results = {
    'MiniLM':              lambda q: search(q, rerank=False),
    'MiniLM + reranker':   lambda q: search(q, rerank=True),
    'E5-small-v2':         lambda q: search_e5(q, rerank=False),
    'E5-small-v2 + reranker': lambda q: search_e5(q, rerank=True),
}
pd.Series({name: evaluate_fn(fn) for name, fn in results.items()}, name='NDCG@10').round(4)

MiniLM                    0.7031
MiniLM + reranker         0.7597
E5-small-v2               0.7425
E5-small-v2 + reranker    0.7621
Name: NDCG@10, dtype: float64

## Late Interaction, Learned Sparse, Generative Index
Сравниваем с тем, что было выше, на одной метрике (NDCG@10).

Для Generative Index нужно обучение, поэтому запросы делим на train (75%) и test (25%): модель учится только на train-запросах. Остальные методы обучения не требуют, их считаем на всех запросах, а в сравнении смотрим и test-часть.

In [15]:
rng = np.random.RandomState(42)
test_qids = set(rng.choice(query['query_id'], int(len(query) * 0.25), replace=False))
train_qids = set(query['query_id']) - test_qids
qtext = dict(zip(query['query_id'], query['query']))
cols = ['product_id', 'product_name', 'product_class', 'average_rating']
len(train_qids), len(test_qids)

(360, 120)

## Late Interaction: ColBERT
Каждый токен текста получает свой вектор. Для токена запроса берём максимальную близость к токенам товара (MaxSim) и суммируем по всем токенам запроса.

In [16]:
import string
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

CB, L = 'lightonai/colbertv2.0', 128
dtype = torch.float16 if device == 'cuda' else torch.float32

cb_tok = AutoTokenizer.from_pretrained(CB)
cb_bert = AutoModel.from_pretrained(CB).to(device).eval()
cb_lin = torch.nn.Linear(768, 128, bias=False).to(device).eval()
cb_lin.weight.data = load_file(hf_hub_download(CB, '1_Dense/model.safetensors'))['linear.weight'].to(device)
skip_ids = torch.tensor([i for i in cb_tok.convert_tokens_to_ids(list(string.punctuation)) if i != cb_tok.unk_token_id])

@torch.no_grad()
def cb_encode(texts, is_query, max_len):
    b = cb_tok(texts, padding='max_length' if is_query else True, truncation=True, max_length=max_len - 1, return_tensors='pt')
    ids, att = b['input_ids'], b['attention_mask']
    ids = torch.cat([ids[:, :1], torch.full_like(ids[:, :1], 1 if is_query else 2), ids[:, 1:]], 1)
    att = torch.cat([att[:, :1], att[:, :1], att[:, 1:]], 1)
    if is_query:
        ids = ids.masked_fill(att == 0, cb_tok.mask_token_id)
    out = cb_lin(cb_bert(input_ids=ids.to(device), attention_mask=att.to(device)).last_hidden_state)
    keep = torch.ones_like(att, dtype=torch.bool) if is_query else att.bool() & ~torch.isin(ids, skip_ids)
    return torch.nn.functional.normalize(out, dim=-1), keep.to(device)

if os.path.exists('colbert_docs.pt'):
    doc_t, doc_mask = torch.load('colbert_docs.pt')
else:
    doc_t = torch.zeros(len(product), L, 128, dtype=torch.float16)
    doc_mask = torch.zeros(len(product), L, dtype=torch.bool)
    texts = product['text'].tolist()
    for i in tqdm(range(0, len(texts), 64)):
        e, k = cb_encode(texts[i:i + 64], False, L)
        doc_t[i:i + len(e), :e.shape[1]] = e.half().cpu()
        doc_mask[i:i + len(e), :e.shape[1]] = k.cpu()
    torch.save((doc_t, doc_mask), 'colbert_docs.pt')

doc_t, doc_mask = doc_t.to(device=device, dtype=dtype), doc_mask.to(device)
doc_t.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

torch.Size([42994, 128, 128])

In [17]:
@torch.no_grad()
def search_colbert(q, top_n=10, bs=4096):
    qe = cb_encode([q], True, 32)[0][0].to(dtype)
    scores = torch.cat([(doc_t[i:i + bs] @ qe.T).masked_fill(~doc_mask[i:i + bs, :, None], -1e4).amax(1).sum(1)
                        for i in range(0, len(doc_t), bs)])
    return product.iloc[scores.topk(top_n).indices.cpu().numpy()][cols]

search_colbert('salon chair')

,product_id,product_name,product_class,average_rating
25431,25431,barberpub salon massage chair,Massage Chairs,5.0
7468,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,5.0
24010,24010,bar salon task chair,,
7465,7465,hair salon chair,Massage Chairs|Recliners,4.5
24006,24006,genia bar salon task chair,Office Chairs,
24009,24009,gerster bar salon task chair,,
24008,24008,germanton bar salon task chair,,
42330,42330,gelt fashion casual beauty salon task chair,,
18273,18273,fashion casual office salon task chair,,
18275,18275,evelinda fashion casual office salon task chair,,


## Learned Sparse: SPLADE
BERT предсказывает для текста веса слов из всего словаря (30522 слов), включая слова, которых в тексте нет (расширение). Получается разреженный вектор, поиск идёт как в обычном поиске по словам, только веса выучены.

In [18]:
import scipy.sparse as sp
from transformers import AutoTokenizer, AutoModelForMaskedLM

splade_tok = AutoTokenizer.from_pretrained('naver/splade-cocondenser-ensembledistil')
splade = AutoModelForMaskedLM.from_pretrained('naver/splade-cocondenser-ensembledistil').to(device).eval()

@torch.no_grad()
def splade_encode(texts, bs=32, max_len=128):
    order = np.argsort([len(t) for t in texts])
    rows, cols_, vals = [], [], []
    for i in tqdm(range(0, len(texts), bs), disable=len(texts) <= bs):
        idx = order[i:i + bs]
        b = splade_tok([texts[j] for j in idx], padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
            logits = splade(**b).logits
        v = torch.log1p(torch.relu(logits.masked_fill(b['attention_mask'][..., None] == 0, 0).amax(1).float()))
        r, c = v.nonzero(as_tuple=True)
        rows.append(idx[r.cpu().numpy()]); cols_.append(c.cpu().numpy()); vals.append(v[r, c].cpu().numpy())
    return sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols_))),
                         shape=(len(texts), splade.config.vocab_size))

if os.path.exists('splade_docs.npz'):
    doc_sparse = sp.load_npz('splade_docs.npz')
else:
    doc_sparse = splade_encode(product['text'].tolist())
    sp.save_npz('splade_docs.npz', doc_sparse)
doc_sparse.shape, doc_sparse.nnz / doc_sparse.shape[0]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


((42994, 30522), 132.86207377773644)

In [19]:
def search_splade(q, top_n=10):
    scores = (doc_sparse @ splade_encode([q]).T).toarray().ravel()
    idx = np.argsort(-scores)[:top_n]
    return product.iloc[idx][cols]

search_splade('salon chair')

,product_id,product_name,product_class,average_rating
25431,25431,barberpub salon massage chair,Massage Chairs,5.0
22463,22463,hard floor straight rectangular chair mat,Chairmats,5.0
36889,36889,beveled rectangular chair mat,Chairmats,5.0
25432,25432,barberpub hydraulic salon spa reclining massag...,Massage Chairs|Recliners,4.0
7465,7465,hair salon chair,Massage Chairs|Recliners,4.5
39461,39461,professional salon reclining massage chair,Massage Chairs,
30162,30162,hard floor rectangular chair mat,Chairmats,4.5
7468,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,5.0
25433,25433,barberpub hydraulic reclining massage chair,Massage Chairs|Recliners,4.5
22462,22462,hard floor straight round chair mat,Chairmats,2.5


## Generative Index (DSI)
Модель T5 учится сама генерировать идентификатор товара по тексту запроса, без поиска по векторам.

- Идентификатор товара: путь по иерархической кластеризации (k-means) эмбеддингов E5, например `<id_5><id_17><id_3>`. Похожие товары получают близкие идентификаторы.
- Обучение: (название / описание товара → его id) для всех товаров плюс (запрос → id) по разметке train-запросов.
- Генерация ограничена деревом существующих id, поэтому модель не может выдать несуществующий товар.

In [20]:
from sklearn.cluster import KMeans
from transformers import T5ForConditionalGeneration

K = 32

def cluster_ids(rows, emb):
    if len(rows) <= K:
        return {r: [j] for j, r in enumerate(rows)}
    labels = KMeans(K, n_init=1, random_state=0).fit_predict(emb[rows])
    if len(set(labels)) == 1:
        labels = np.arange(len(rows)) % K
    out = {}
    for c in np.unique(labels):
        for r, code in cluster_ids(rows[labels == c], emb).items():
            out[r] = [int(c)] + code
    return out

docid_codes = cluster_ids(np.arange(len(product)), product_emb_e5)

dsi_tok = AutoTokenizer.from_pretrained('t5-small')
dsi_tok.add_tokens([f'<id_{i}>' for i in range(K)])
dsi = T5ForConditionalGeneration.from_pretrained('t5-small')
dsi.resize_token_embeddings(len(dsi_tok))

code_tok = [dsi_tok.convert_tokens_to_ids(f'<id_{i}>') for i in range(K)]
doc_seq = {r: [code_tok[c] for c in codes] + [dsi_tok.eos_token_id] for r, codes in docid_codes.items()}
seq_row = {tuple(s): r for r, s in doc_seq.items()}
max(len(s) for s in doc_seq.values()), len(seq_row)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


(6, 42994)

In [21]:
row_of = dict(zip(product['product_id'], range(len(product))))
rel = label[label['query_id'].isin(train_qids) & (label['label'] != 'Irrelevant')]
rel = rel[rel['product_id'].isin(row_of)]

inputs = (product['product_name'] + '. ' + product['product_class']).tolist() + rel['query_id'].map(qtext).tolist()
rows = np.concatenate([np.arange(len(product)), rel['product_id'].map(row_of).values])

seq_len = max(len(s) for s in doc_seq.values())
seq_arr = torch.full((len(product), seq_len), -100)
for r, s in doc_seq.items():
    seq_arr[r, :len(s)] = torch.tensor(s)

enc = dsi_tok(inputs, max_length=32, truncation=True, padding='max_length', return_tensors='pt')
X, A, Y = enc['input_ids'].to(device), enc['attention_mask'].to(device), seq_arr[rows].to(device)
lens = enc['attention_mask'].sum(1)
len(inputs)

169335

In [22]:
from tqdm.auto import tqdm

EPOCHS, BS, AMP = 1, 256, device == 'cuda'
dsi.to(device).train()
opt = torch.optim.AdamW(dsi.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler(enabled=AMP)

for ep in range(EPOCHS):
    batches = torch.argsort(lens + torch.rand(len(lens))).split(BS)
    total = 0
    for k in tqdm(torch.randperm(len(batches)).tolist(), desc=f'epoch {ep + 1}'):
        b = batches[k]
        m = int(lens[b].max())
        b = b.to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=AMP):
            loss = dsi(input_ids=X[b, :m], attention_mask=A[b, :m], labels=Y[b]).loss
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()
        total += loss.detach() * len(b)
    print(f'epoch {ep + 1}: loss {total.item() / len(lens):.4f}')
dsi.eval();

epoch 1:   0%|          | 0/662 [00:00<?, ?it/s]

epoch 1: loss 3.7665


In [23]:
trie = {}
for s in doc_seq.values():
    for i in range(len(s)):
        trie.setdefault(tuple(s[:i]), set()).add(s[i])
trie = {k: list(v) for k, v in trie.items()}

def allowed(batch_id, ids):
    return trie.get(tuple(ids[1:].tolist()), [dsi_tok.eos_token_id])

@torch.no_grad()
def search_dsi(q, top_n=10):
    x = dsi_tok(q, return_tensors='pt', truncation=True, max_length=48).to(device)
    out = dsi.generate(**x, num_beams=top_n, num_return_sequences=top_n, max_new_tokens=8, prefix_allowed_tokens_fn=allowed)
    rows = []
    for s in out[:, 1:].tolist():
        s = s[:s.index(dsi_tok.eos_token_id) + 1] if dsi_tok.eos_token_id in s else s
        if tuple(s) in seq_row:
            rows.append(seq_row[tuple(s)])
    return product.iloc[rows][cols]

search_dsi('salon chair')

,product_id,product_name,product_class,average_rating
1616,1616,fashion casual lift chair office work chair be...,,
3670,3670,stackhouse executive desk,Desks,4.5
2813,2813,coole task chair,Office Chairs,4.0
13185,13185,bronze design chair mat,Chairmats,4.5
18271,18271,eveliina fashion casual beauty salon task chair,Office Chairs,
3619,3619,wisconsin solid wood desk,Desks,5.0
5298,5298,krauthamer task chair,Office Chairs,
23731,23731,nebel desk,Desks,4.5
2969,2969,electric height adjustable standing desk conve...,Commercial Desks|Desks,4.5
5555,5555,amiracle executive chair,Office Chairs,3.0


## Итоговое сравнение

In [ ]:
def per_query(fn, qids):
    return {qid: ndcg(fn(qtext[qid])['product_id'].tolist(), gain.get(qid, {})) for qid in qids}

methods = {
    'MiniLM (bi-encoder)':        lambda q: search(q, rerank=False),
    'MiniLM + reranker':          lambda q: search(q, rerank=True),
    'E5-small-v2':                lambda q: search_e5(q, rerank=False),
    'E5-small-v2 + reranker':     lambda q: search_e5(q, rerank=True),
    'ColBERT (late interaction)': search_colbert,
    'SPLADE (learned sparse)':    search_splade,
    'DSI (generative index)':     search_dsi,
}
scores = {n: per_query(fn, test_qids if n.startswith('DSI') else qtext) for n, fn in methods.items()}

pd.DataFrame({
    'NDCG@10 test-запросы': {n: np.mean([s[q] for q in test_qids]) for n, s in scores.items()},
    'NDCG@10 все запросы':  {n: np.mean(list(s.values())) if len(s) == len(qtext) else np.nan for n, s in scores.items()},
}).round(4)

,NDCG@10 test-запросы,NDCG@10 все запросы
MiniLM (bi-encoder),0.7023,0.7031
MiniLM + reranker,0.7660,0.7597
E5-small-v2,0.7377,0.7425
E5-small-v2 + reranker,0.7660,0.7621
ColBERT (late interaction),0.7444,0.7377
SPLADE (learned sparse),0.7306,0.7327
DSI (generative index),0.0412,NaN


DSi почти не обучилась, поскольку на обучении было всего 1 эпоха. Для улучшения результатов можно увеличить количество эпох, размер батча. Но на это не было времени, она и так долго считала 